# BLHS Reasoning Graph Notebook

Notebook này dùng `neo4j_reasoning_graph_helper.py` để:
- materialize `ReasoningCase`, `ReasoningTerm`, `ReasoningCandidate`, `ReasoningStep`,
- xếp hạng ứng viên theo score mạnh hơn,
- truy vết vì sao điều luật được chọn,
- truy ra điều luật nào bị loại trừ bởi ứng viên đó.

In [1]:
import pandas as pd

from neo4j_reasoning_graph_helper import (
    build_reasoning_graph,
    get_graph_stats,
    materialize_normalized_properties,
    setup_reasoning_graph,
    trace_case,
    trace_exclusions,
)

DISPLAY_CANDIDATE_COLUMNS = [
    'level', 'so_dieu', 'tieu_de', 'final_score',
    'support_score', 'missing_count', 'exception_count',
    'excluded_articles'
]
DISPLAY_TRACE_COLUMNS = [
    'rank', 'status', 'so_dieu', 'tieu_de',
    'final_score', 'exception_count', 'excluded_articles'
]


def safe_frame(rows, columns):
    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=columns)
    for col in columns:
        if col not in df.columns:
            df[col] = None
    return df[columns]

In [2]:
setup_reasoning_graph()
normalized_count = materialize_normalized_properties()
stats = get_graph_stats()

print('Da san sang reasoning graph.')
print(f"So node Dieu hien co: {stats['dieu_count']}")
print(f"So node/field duoc bo sung normalize: {normalized_count}")

Da san sang reasoning graph.
So node Dieu hien co: 427
So node/field duoc bo sung normalize: 10034


In [29]:
case_id, candidate_rows = build_reasoning_graph(
    case_name='Hiep dam nguoi duoi 16 tuoi',
    action_text='làm giả',
    victim_text='',
    consequence_text='',
    actor_text='',
    intent_text='',
    top_k=10,
)

print(f'Case ID: {case_id}')
print(f'So ung vien: {len(candidate_rows)}')

if not candidate_rows:
    print('Khong tim thay ung vien phu hop. Thu bo bot intent/consequence hoac doi cach dien dat ngan hon.')

safe_frame(candidate_rows, DISPLAY_CANDIDATE_COLUMNS).head(10)


Case ID: case-20260404-090146-841270f1
So ung vien: 2


,level,so_dieu,tieu_de,final_score,support_score,missing_count,exception_count,excluded_articles
0,Khoan,212.0,"Tội làm giả tài liệu trong hồ sơ chào bán, niê...",13.0,13,0,0,[]
1,Khoan,341.0,"Tội làm giả con dấu, tài liệu của cơ quan, tổ ...",13.0,13,0,0,[]


In [30]:
trace_df = trace_case(case_id)

if trace_df.empty:
    print('Chua co trace cho case nay.')

safe_frame(trace_df.to_dict(orient='records'), DISPLAY_TRACE_COLUMNS).head(10)


,rank,status,so_dieu,tieu_de,final_score,exception_count,excluded_articles
0,1,RECOMMENDED,212.0,"Tội làm giả tài liệu trong hồ sơ chào bán, niê...",13.0,0,[]
1,2,CANDIDATE,341.0,"Tội làm giả con dấu, tài liệu của cơ quan, tổ ...",13.0,0,[]


In [31]:
exclusion_df = trace_exclusions(case_id, top_k=5)
if exclusion_df.empty:
    print('Khong co dieu luat nao bi loai tru trong top ung vien hien tai.')
exclusion_df

Khong co dieu luat nao bi loai tru trong top ung vien hien tai.


""


In [32]:
if not trace_df.empty and 'steps' in trace_df.columns:
    top1_steps = trace_df.iloc[0]['steps']
    for step in top1_steps:
        print(step)

{'detail': 'action: semantic=làm giả; text=lam gia; missing=khong co', 'dimension': 'action', 'input_terms': ['lam gia'], 'matched_terms': ['làm giả'], 'missing_terms': [], 'step_type': 'SUPPORT', 'text_hits': ['lam gia'], 'verdict': 'MATCH', 'weight': 5}
{'detail': 'Loai tru dieu: khong co', 'dimension': 'exclusion', 'input_terms': [], 'matched_terms': [], 'missing_terms': [], 'step_type': 'EXCLUSION_SCAN', 'text_hits': [], 'verdict': 'NO_EXCEPTION', 'weight': 1}
